In [1]:
# Install OpenNMT-py 3.x
!pip3 install OpenNMT-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.8/262.8 KB 5.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 15.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 35.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 KB 28.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 71.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 KB 38.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 111.3 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 KB 36.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.2/29.2 MB 67.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 5.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 60.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
%pip install "numpy<2"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 117.9 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.2
    Uninstalling numpy-2.1.2:
      Successfully uninstalled numpy-2.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.21.0+cu124 requires torch==2.6.0, but you have torch 2.2.2 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Create the YAML configuration file
# On a regular machine, you can create it manually or with nano
# Note here we are using some smaller values because the dataset is small
# For larger datasets, consider increasing: train_steps, valid_steps, warmup_steps, save_checkpoint_steps, keep_checkpoint

config = '''# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: en-zh_tagged.en.subword.train
        path_tgt: en-zh.zh-filtered.zh.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: en-zh_tagged.en.subword.dev
        path_tgt: en-zh.zh-filtered.zh.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 20000
tgt_vocab_size: 10000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 512
src_seq_length: 512

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.fren

# Stop training if it does not imporve after n validations
early_stopping: 4

# Default: 5000 - Save a model checkpoint for each n
save_checkpoint_steps: 2000

# To save space, limit checkpoints to last n
# keep_checkpoint: 3

seed: 3435

# Default: 100000 - Train the model to max n steps 
# Increase to 200000 or more for large datasets
# For fine-tuning, add up the required steps to the original steps
train_steps: 15000

# Default: 10000 - Run validation after n steps
valid_steps: 2000

# Default: 4000 - for large datasets, try up to 8000
warmup_steps: 4000
report_every: 100

# Number of GPUs, and IDs of GPUs
world_size: 1
gpu_ranks: [0]

# Batching
bucket_size: 262144
num_workers: 0  # Default: 2, set to 0 when RAM out of memory
batch_type: "tokens"
batch_size: 4096   # Tokens per batch, change when CUDA out of memory
valid_batch_size: 2048
max_generator_batches: 2
accum_count: [4]
accum_steps: [0]

# Optimization
model_dtype: "fp16"
optim: "adam"
learning_rate: 2
# warmup_steps: 8000
decay_method: "noam"
adam_beta2: 0.998
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
normalization: "tokens"
weight_decay: 0.0001

# Model
encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 8
hidden_size: 512
word_vec_size: 512
transformer_ff: 2048
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
'''

with open("config.yaml", "w+") as config_yaml:
  config_yaml.write(config)

In [2]:
# Find the number of CPUs/cores on the machine
!nproc --all

64


In [3]:
# Build Vocabulary

# -config: path to your config.yaml file
# -n_sample: use -1 to build vocabulary on all the segment in the training dataset
# -num_threads: change it to match the number of CPUs to run it faster

!onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 64

Corpus corpus_1's weight should be given. We default it to 1 for you.
[2025-04-06 06:23:29,775 INFO] Counter vocab from -1 samples.
[2025-04-06 06:23:29,775 INFO] n_sample=-1: Build vocab on full datasets.
[2025-04-06 06:23:33,765 INFO] * Transform statistics for corpus_1(1.56%):
			* FilterTooLongStats(filtered=3)

[2025-04-06 06:23:33,771 INFO] * Transform statistics for corpus_1(1.56%):
			* FilterTooLongStats(filtered=2)

[2025-04-06 06:23:33,972 INFO] * Transform statistics for corpus_1(1.56%):
			* FilterTooLongStats(filtered=1)

[2025-04-06 06:23:33,991 INFO] * Transform statistics for corpus_1(1.56%):
			* FilterTooLongStats(filtered=1)

[2025-04-06 06:23:34,374 INFO] Counters src: 20114
[2025-04-06 06:23:34,375 INFO] Counters tgt: 11303


In [4]:
# Check if the GPU is active
!nvidia-smi -L

GPU 0: NVIDIA GeForce RTX 3090 (UUID: GPU-d28ff538-6ab5-ca99-63a4-87f215425b3f)


In [5]:
# Check if the GPU is visable to PyTorch
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

gpu_memory = torch.cuda.mem_get_info(0)
print("Free GPU memory:", gpu_memory[0]/1024**2, "out of:", gpu_memory[1]/1024**2)

True
NVIDIA GeForce RTX 3090
Free GPU memory: 23992.6875 out of: 24252.6875


In [6]:
# Train the NMT model
!onmt_train -config config.yaml

[2025-04-06 06:23:52,967 WARNING] Corpus corpus_1's weight should be given. We default it to 1 for you.
[2025-04-06 06:23:52,967 INFO] Parsed 2 corpora from -data.
[2025-04-06 06:23:52,967 INFO] Get special vocabs from Transforms: {'src': [], 'tgt': []}.
[2025-04-06 06:23:53,000 INFO] The first 10 tokens of the vocabs are:['<unk>', '<blank>', '<s>', '</s>', '_', 'NOUN', 'PUNCT', 'PRON', 'VERB', 'ADP']
[2025-04-06 06:23:53,000 INFO] The decoder start token is: <s>
[2025-04-06 06:23:53,000 INFO] Building model...
[2025-04-06 06:23:53,607 INFO] Switching model to float32 for amp/apex_amp
[2025-04-06 06:23:53,608 INFO] Non quantized layer compute is fp16
[2025-04-06 06:23:53,763 INFO] NMTModel(
  (encoder): TransformerEncoder(
    (embeddings): Embeddings(
      (make_embedding): Sequential(
        (emb_luts): Elementwise(
          (0): Embedding(20000, 512, padding_idx=1)
        )
        (pe): PositionalEncoding()
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (trans

## Translate

In [7]:
# Translate the "subworded" source file of the test dataset
# Change the model name, if needed.
# gpu
!onmt_translate -model models/model.fren_step_15000.pt -src en-zh_tagged.en.subword.test -output zh.base.translated -gpu 0 -min_length 1



[2025-04-06 07:33:00,003 INFO] Loading checkpoint from models/model.fren_step_15000.pt
[2025-04-06 07:33:00,577 INFO] Loading data into the model
[2025-04-06 07:33:11,861 INFO] PRED SCORE: -0.6793, PRED PPL: 1.97 NB SENTENCES: 2000
Time w/o python interpreter load/terminate:  11.864367961883545


In [8]:
# Check the first 5 lines of the translation file
!head -n 30 zh.base.translated

▁ 禁止 在 捷 克 、 东 德国 、 ▁ 爱 沙 沙 尼亚 、 陶 宛 、 马 里 斯 加 、 马 达 加 斯 加 、 ▁ 普 利 斯 和 塞 尔 维 亚 等 , 我可以 继续 下去 , ▁现在 是 突 尼 西亚 和 埃及 。
▁我不知道 他们 将会 用 这些东西 做些什么 。
▁它 需要 一些 探讨 的概念 ▁因为它 是一个 没有 雕塑 化的 系统 。
▁ 伟大的 老师 这么做 了 , ▁但是 伟大的 老师 也 做的就是 辅 导 , ▁ 激发 , 激发 , 参与 。
▁所以 , 我 只用 插 电 和 包 裹 ▁ 完成 这个 简 易 的任务 , ▁最后 , 在 4 0 0 0 次 的 努力 中 , ▁我 几乎 在 不 卫生 的情况下 失去 , ▁我 找到了 蛋白质 。
▁我 注意到 它们 会 移动 这个 小小的 理论 装置 , ▁以 一 度 或 二 度 的 温度 变化 。
▁但这 是 后果 。
▁所以 “ 第二 人生 ” 的 平均 年龄 是 3 2 岁 , ▁而 “ 第二 人生 ” 的 应用 ▁ 大大 增加 了 你的 年龄 增长 。 所以 当你 从 3 0 岁 到 6 0 岁 , ▁他们 有很多 人 在 “ 第二 人生 ” 里 使用 了 —— ▁这 还 不是 很 尖 锐 的 曲线 —— ▁非常 分散 的 使用 —— ▁ 到 每周 4 0 % , 也就是 4 0 %。
▁所以 它 不停地 想 清楚 它的 抽象 概念 。
▁ 这项技术 的 好处 是 ▁ 能让 手机 开始 看到 和 理解 ▁人类 大脑 是如何 运作 的 。
▁ 借助 一个 建筑师 的 帮助 , ▁ 居民 真的 从 地面 提高 过来 。
▁然后 学生们 会 进入 我们的 音 响 室 , ▁ 发出 自己的 说 唱 诗 , ▁ 唱 着 自己的 节奏 。
▁有时 , 我 从 冒险 者 的 教堂 周 日 ▁ 抚 养 了 小 老 太 太 太 太 太 太 太 太 太 太 太 太 , ▁ 给我 看 这些 关于 天堂 的 漫画 。
▁如果你 从 小 农民 那里 ▁ 填 满 当地 农业 的 杯子 , ▁你会 有 巨大的 变化 。
▁然后 , 她 在 一 小时 后 找到 他 , 说 , ▁“ 你 是谁 ?”
▁ 很 显然 , 我要 把 那 副 牌 组装 起来 。
▁它 是一种 综合 的 系统 , ▁尽管 所有的 计划 都是 

In [10]:
# If needed install/update sentencepiece
!pip3 install --upgrade -q sentencepiece

# Desubword the translation file
!python3 ./3-desubword.py ./target.model zh.base.translated

Done desubwording! Output: zh.base.translated.desubword


In [12]:
# Desubword the target file (reference) of the test dataset
# Note: You might as well have split files *before* subwording during dataset preperation, 
# but sometimes datasets have tokeniztion issues, so this way you are sure the file is really untokenized.
!python3 ./MT-Preparation/subwording/3-desubword.py ./source.model en-zh_tagged.en.subword.test

# Desubword the test file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model en-zh.zh-filtered.zh.subword.test

Done desubwording! Output: en-zh_tagged.en.subword.test.desubword
Done desubwording! Output: en-zh.zh-filtered.zh.subword.test.desubword


In [13]:
# Check the first 5 lines of the desubworded translation file
!head -n 30 zh.base.translated.desubword

禁止在捷克、东德国、 爱沙沙尼亚、陶宛、马里斯加、马达加斯加、 普利斯和塞尔维亚等,我可以继续下去, 现在是突尼西亚和埃及。
我不知道他们将会用这些东西做些什么。
它需要一些探讨的概念 因为它是一个没有雕塑化的系统。
伟大的老师这么做了, 但是伟大的老师也做的就是辅导, 激发,激发,参与。
所以,我只用插电和包裹 完成这个简易的任务, 最后,在4000次的努力中, 我几乎在不卫生的情况下失去, 我找到了蛋白质。
我注意到它们会移动这个小小的理论装置, 以一度或二度的温度变化。
但这是后果。
所以“第二人生”的平均年龄是32岁, 而“第二人生”的应用 大大增加了你的年龄增长。所以当你从30岁到60岁, 他们有很多人在“第二人生”里使用了—— 这还不是很尖锐的曲线—— 非常分散的使用—— 到每周40%,也就是40%。
所以它不停地想清楚它的抽象概念。
这项技术的好处是 能让手机开始看到和理解 人类大脑是如何运作的。
借助一个建筑师的帮助, 居民真的从地面提高过来。
然后学生们会进入我们的音响室, 发出自己的说唱诗, 唱着自己的节奏。
有时,我从冒险者的教堂周日 抚养了小老太太太太太太太太太太太太, 给我看这些关于天堂的漫画。
如果你从小农民那里 填满当地农业的杯子, 你会有巨大的变化。
然后,她在一小时后找到他,说, “你是谁?”
很显然,我要把那副牌组装起来。
它是一种综合的系统, 尽管所有的计划都是如此。
然后现代人在非洲的某处出现, 走出非洲,就应该是在中东。
这些破坏让我们解决问题, 它们帮助我们变得更有创造力。
格伦长得老了。
如果可以的话,那不是很棒吗? 第一次,双眼 可以完全适合你, 并且不需要任何安装链子, 所以机遇会自动化。
中国的增长是重大变化的一部分, 从根本上的变化,也就是25年前, 发展中国家,世界上最穷的国家, 尽管他们占了世界人口的很大一部分, 他们占据了世界产出的三分之一。
当然了,现在我家里非常敏感 当我们把灯熄灭的时候
最低绩效的国家,中央非洲人, 31分。
这些都是自然的人类冲动, 但因为科技, 在这些冲动上行动, 只有一次点击。
像这是我第一次 在水中跳入水中 第一次在西班牙的地中海滨时 见到的。
这在世界杯赛上玩耍。
我们确实找到了。
我知道,消防员们告诉我, 这不是罕见的。
不,我想他说的是有选择性的森林。


## Evaluation

In [14]:
# Download the BLEU script
!wget https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py

--2025-04-06 07:35:30--  https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
200 OKequest sent, awaiting response... 
Length: 957 [text/plain]
Saving to: ‘compute-bleu.py’

compute-bleu.py     100%[===================>]     957  --.-KB/s    in 0s      

2025-04-06 07:35:30 (34.9 MB/s) - ‘compute-bleu.py’ saved [957/957]



In [15]:
# Install sacrebleu
!pip3 install sacrebleu

In [16]:
# Evaluate the translation (without subwording)
!python3 compute-bleu.py en-zh.zh-filtered.zh.subword.test.desubword zh.base.translated.desubword

Reference 1st sentence: 在捷克斯洛伐克,东德 爱沙尼亚,拉脱维亚,立陶宛, 马里,马达加斯加, 波兰,菲律宾, 塞尔维亚,斯洛维尼亚的独裁政府,我可以继续, 还有现在的突尼斯和埃及。
MTed 1st sentence: 禁止在捷克、东德国、 爱沙沙尼亚、陶宛、马里斯加、马达加斯加、 普利斯和塞尔维亚等,我可以继续下去, 现在是突尼西亚和埃及。
BLEU:  2.8760881771959723
